Fine-tuning:
https://huggingface.co/docs/transformers/tasks/sequence_classification
https://huggingface.co/docs/transformers/en/training
https://huggingface.co/docs/transformers/en/models

In [ ]:
from huggingface_hub import login

login()

In [4]:
import pandas as pd

df = pd.read_csv("phishing_dataset.csv")

In [5]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [6]:
from datasets import load_dataset
from transformers import AutoTokenizer


dataset = load_dataset("csv", data_files="phishing_dataset.csv")["train"]

dataset = dataset.train_test_split(test_size=0.1)

def tokenize(batch):
    return tokenizer(
        batch["email"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

tokenized_dataset = dataset.map(tokenize, batched=True)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/180 [00:00<?, ? examples/s]

Map:   0%|          | 0/21 [00:00<?, ? examples/s]

In [7]:
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True)

In [8]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [9]:
import numpy as np


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

Training


In [10]:
id2label = {0: "SAFE", 1: "PHISHING"}
label2id = {"SAFE": 0, "PHISHING": 1}

In [11]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
!pip install evaluate

In [13]:
import evaluate

accuracy = evaluate.load("accuracy")

In [14]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="phishing_model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True
)
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.548109,0.857143
2,No log,0.450018,0.952381


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=24, training_loss=0.5678491989771525, metrics={'train_runtime': 32.926, 'train_samples_per_second': 10.934, 'train_steps_per_second': 0.729, 'total_flos': 47688263516160.0, 'train_loss': 0.5678491989771525, 'epoch': 2.0})

In [15]:
trainer.save_model("phishing_model")
tokenizer.save_pretrained("phishing_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('phishing_model/tokenizer_config.json', 'phishing_model/tokenizer.json')

Walk through:


In [16]:
from transformers import AutoModelForSequenceClassification
import torch

In [17]:
text = "Hey, just confirming our meeting is still scheduled for 3pm tomorrow. Let me know if anything changes."

In [18]:

tokenizer = AutoTokenizer.from_pretrained("phishing_model")
model = AutoModelForSequenceClassification.from_pretrained("phishing_model")
inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)

with torch.no_grad():
    logits = model(**inputs).logits

pred = torch.argmax(logits).item()

print("Final prediction: ", pred)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Final prediction:  1
